In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, top_k_accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import lightgbm as lgb
import pickle

In [3]:
df=pd.read_csv(r"D:\ml_udemy\00 ML CODE\Projects\Digital Diagnosis v2\data\disease_dataset_clean.csv")

In [4]:
df.head()

,disease,agitation,angina_pectoris,arm_weakness,ascites,asthenia,calf_pain_walking,chest_tightness,chill,confusion,...,sore_throat,tremor,unresponsiveness,vomiting,weepiness,weight_gain,wheezing,worry,yellow_eyes,yellow_skin
0,hypertensive disease,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,hypertensive disease,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,hypertensive disease,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,hypertensive disease,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,hypertensive disease,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [5]:
x=df.drop("disease", axis=1)
y=df['disease']

In [6]:
le=LabelEncoder()
y=le.fit_transform(y)

In [7]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, stratify=y, random_state=42)

In [8]:
model = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    num_leaves=127,
    min_child_samples=10,
    feature_fraction=0.5,
    bagging_fraction=0.8,
    bagging_freq=5,
    max_depth=20,
    reg_alpha=0.5,
    reg_lambda=0.5,
    class_weight='balanced',
    random_state=42,
    verbose=-1,
    n_jobs=-1
)

In [9]:
model.fit(x_train, y_train)

LGBMClassifier(bagging_fraction=0.8, bagging_freq=5, class_weight='balanced',
               feature_fraction=0.5, learning_rate=0.05, max_depth=20,
               min_child_samples=10, n_estimators=1000, n_jobs=-1,
               num_leaves=127, random_state=42, reg_alpha=0.5, reg_lambda=0.5,
               verbose=-1)

In [10]:
y_pred=model.predict(x_test)
y_prob = model.predict_proba(x_test)

In [11]:
print("Top-1:", accuracy_score(y_test, y_pred))
print("Top-3:", top_k_accuracy_score(y_test, y_prob, k=3))
print("Top-5:", top_k_accuracy_score(y_test, y_prob, k=5))

print("Macro Precision:", precision_score(y_test, y_pred, average='macro'))
print("Macro Recall:", recall_score(y_test, y_pred, average='macro'))
print("Macro F1:", f1_score(y_test, y_pred, average='macro'))
print("Weighted F1:", f1_score(y_test, y_pred, average='weighted'))

print("\nFull Classification Report:\n")
print(classification_report(y_test, y_pred))

Top-1: 0.8582834331337326
Top-3: 0.9860279441117764
Top-5: 0.9960079840319361
Macro Precision: 0.8728556443556443
Macro Recall: 0.8587272727272726
Macro F1: 0.8588896706036967
Weighted F1: 0.8585725255525914

Full Classification Report:

              precision    recall  f1-score   support

           0       0.91      1.00      0.95        10
           1       1.00      0.70      0.82        10
           2       0.89      0.80      0.84        10
           3       0.90      0.90      0.90        10
           4       1.00      1.00      1.00        10
           5       1.00      0.80      0.89        10
           6       0.69      0.90      0.78        10
           7       0.78      0.64      0.70        11
           8       0.89      0.80      0.84        10
           9       0.89      0.80      0.84        10
          10       1.00      0.90      0.95        10
          11       0.82      0.90      0.86        10
          12       0.73      0.80      0.76        10
     

In [12]:
with open(r"D:\ml_udemy\00 ML CODE\Projects\Digital Diagnosis v2\models\lightgbm_model.pkl", "wb") as f:
    pickle.dump(model, f)

with open(r"D:\ml_udemy\00 ML CODE\Projects\Digital Diagnosis v2\models\label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

print("Model saved!")

Model saved!


In [13]:
def predict_top3(symptom_values: list):
    """Pass a list of 0/1 values matching your symptom columns."""
    x = np.array(symptom_values).reshape(1, -1)
    probs = model.predict_proba(x)[0]
    top3_idx = np.argsort(probs)[::-1][:3]

    print("Top 3 Predicted Diseases:")
    for rank, idx in enumerate(top3_idx, 1):
        disease = le.inverse_transform([idx])[0]
        prob = probs[idx] * 100
        print(f"  {rank}. {disease:20s} {prob:.2f}%")

In [14]:
sample_input = [0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
predict_top3(sample_input)

Top 3 Predicted Diseases:
  1. migraine disorders   94.20%
  2. upper respiratory infection 1.78%
  3. gastroesophageal reflux disease 1.46%
